In [1]:
import torch
import torch.nn as nn
from torchvision import models
from backbone import BaseCNNModel

from cvtools.models.pytorch import PyTorchModel, L2Norm

/home/julian/Projects/KEMAI/iconic-visual-search/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/julian/Projects/KEMAI/iconic-visual-search/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or b

In [ ]:
vgg16 = models.vgg16(weights="IMAGENET1K_V1")
print(vgg16)

features = nn.Sequential(
    *list(vgg16.features)[:30],  # All conv layers from pretrained VGG until layer 30
    vgg16.avgpool,
    # nn.AdaptiveAvgPool2d((1, 1)),  # Global average pooling
    nn.Flatten()
)
print(features)

features = nn.Sequential(
    *list(vgg16.children())[0][:30],  # All conv layers from pretrained VGG until layer 30
    # nn.AdaptiveAvgPool2d((1, 1)),  # Global average pooling
    # nn.Flatten()
)
print(features)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
class VGG(BaseCNNModel):

    def __init__(self, n_classes, classifier="linear", pretrained=True, output_layers=[]):
        # VGG16 outputs 512 features after global average pooling
        super().__init__(embedding_dim=512, n_classes=n_classes, classifier=classifier)

        # Load pretrained VGG16
        vgg16 = models.vgg16(weights="IMAGENET1K_V1")

        # Extract features (all convolutional layers)
        # We'll modify the last part to add global average pooling
        self.features = nn.Sequential(
            *list(vgg16.features)[:30],  # All conv layers from pretrained VGG until layer 30
            nn.AdaptiveAvgPool2d((1, 1)),  # Global average pooling
            nn.Flatten()
        )

        if classifier == "arcface":
            self.features.append(L2Norm())

        self._register_hooks(output_layers)



self.features = nn.Sequential(*list(model.children())[0][:30])
      for param in self.features.parameters():
        param.requires_grad_ = False